# Joint training: BERT on SQuAD v1.1 + NewsQA combined

Trains a fresh `bert-base-uncased` on SQuAD v1.1 and the NewsQA subset combined into a single training set from the start, as a comparison point against the SQuAD-only baseline and the sequential fine-tune (see `bert_baseline_results.md` and `newsqa_finetune_results.md`).

Requires `train-newsqa.json` and `dev-newsqa.json` to already exist (run `preprocess_newsqa.ipynb` first if not).

The resulting model is saved to its own folder, `qa_model_joint`, separate from `qa_model` and `qa_model_newsqa`.

In [1]:
!pip install -q -U transformers datasets accelerate evaluate

import transformers
print("transformers version:", transformers.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
transformers version: 5.20.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import collections
import subprocess
import dataclasses
import inspect

import numpy as np
import torch
from datasets import Dataset, DatasetDict, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

MODEL_CHECKPOINT = "bert-base-uncased"   # fresh model, trained on both datasets from the start

PROJECT_DIR = "/content/drive/MyDrive/CS4248 Group Project"
DATA_DIR = os.path.join(PROJECT_DIR, "qa")

SQUAD_TRAIN_FILE = os.path.join(DATA_DIR, "train-v1.1.json")
SQUAD_DEV_FILE = os.path.join(DATA_DIR, "dev-v1.1.json")
NEWSQA_TRAIN_FILE = os.path.join(DATA_DIR, "train-newsqa.json")
NEWSQA_DEV_FILE = os.path.join(DATA_DIR, "dev-newsqa.json")
EVAL_SCRIPT = os.path.join(DATA_DIR, "evaluate-v2.0.py")

MAX_LENGTH = 384
DOC_STRIDE = 128
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 3e-5   # same as the original SQuAD-only baseline, since this also trains from scratch
NUM_EPOCHS = 2
N_BEST = 20
MAX_ANSWER_LENGTH = 30
RANDOM_SEED = 42

OUTPUT_DIR = os.path.join(PROJECT_DIR, "qa_model_joint")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PREDICTIONS_SQUAD_FILE = os.path.join(DATA_DIR, "predictions_squad_after_joint.json")
PREDICTIONS_NEWSQA_FILE = os.path.join(DATA_DIR, "predictions_newsqa_after_joint.json")

MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

In [4]:
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > Hardware "
    "accelerator > GPU (T4), save, reconnect, then re-run from the top."
)
print("GPU:", torch.cuda.get_device_name(0))

for path in (SQUAD_TRAIN_FILE, SQUAD_DEV_FILE, NEWSQA_TRAIN_FILE, NEWSQA_DEV_FILE, EVAL_SCRIPT):
    assert os.path.exists(path), (
        f"Missing: {path}\n"
        "If this is train-newsqa.json / dev-newsqa.json, run preprocess_newsqa.ipynb first."
    )
print("All required files found.")

GPU: Tesla T4
All required files found.


In [5]:
def load_squad_json(path):
    with open(path) as f:
        squad = json.load(f)
    ids, questions, contexts, answers = [], [], [], []
    for article in squad["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                ids.append(qa["id"])
                questions.append(qa["question"])
                contexts.append(context)
                answers.append(
                    {
                        "text": [a["text"] for a in qa["answers"]],
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                    }
                )
    return Dataset.from_dict(
        {"id": ids, "question": questions, "context": contexts, "answers": answers}
    )


print("Loading SQuAD train/dev and NewsQA train/dev...")
squad_train = load_squad_json(SQUAD_TRAIN_FILE)
squad_dev_raw = load_squad_json(SQUAD_DEV_FILE)
newsqa_train = load_squad_json(NEWSQA_TRAIN_FILE)
newsqa_dev_raw = load_squad_json(NEWSQA_DEV_FILE)

print(f"SQuAD train: {len(squad_train)}, NewsQA train: {len(newsqa_train)}")

combined_train = concatenate_datasets([squad_train, newsqa_train]).shuffle(seed=RANDOM_SEED)
print(f"Combined train (SQuAD + NewsQA): {len(combined_train)}")

if MAX_TRAIN_SAMPLES is not None:
    combined_train = combined_train.select(range(MAX_TRAIN_SAMPLES))
if MAX_EVAL_SAMPLES is not None:
    squad_dev_raw = squad_dev_raw.select(range(MAX_EVAL_SAMPLES))
    newsqa_dev_raw = newsqa_dev_raw.select(range(MAX_EVAL_SAMPLES))

Loading SQuAD train/dev and NewsQA train/dev...
SQuAD train: 87599, NewsQA train: 10003
Combined train (SQuAD + NewsQA): 97602


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
assert tokenizer.is_fast, "Need a fast tokenizer for offset mapping"

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = tokenized.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


def prepare_validation_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    example_ids = []
    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    tokenized["example_id"] = example_ids
    return tokenized


print("Tokenizing combined train set...")
train_dataset = combined_train.map(
    prepare_train_features,
    batched=True,
    remove_columns=combined_train.column_names,
)

print("Tokenizing SQuAD validation set...")
squad_validation_dataset = squad_dev_raw.map(
    prepare_validation_features,
    batched=True,
    remove_columns=squad_dev_raw.column_names,
)

print("Tokenizing NewsQA validation set...")
newsqa_validation_dataset = newsqa_dev_raw.map(
    prepare_validation_features,
    batched=True,
    remove_columns=newsqa_dev_raw.column_names,
)

Tokenizing combined train set...


Map:   0%|          | 0/97602 [00:00<?, ? examples/s]

Tokenizing SQuAD validation set...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Tokenizing NewsQA validation set...


Map:   0%|          | 0/1003 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)

ta_fields = {f.name for f in dataclasses.fields(TrainingArguments)}


def pick(*name_value_pairs):
    for name, value in name_value_pairs:
        if name in ta_fields:
            return {name: value}
    return {}


ta_kwargs = {"output_dir": OUTPUT_DIR}
ta_kwargs.update(pick(("eval_strategy", "no"), ("evaluation_strategy", "no")))
ta_kwargs.update(pick(("save_strategy", "epoch")))
ta_kwargs["learning_rate"] = LEARNING_RATE
ta_kwargs["per_device_train_batch_size"] = BATCH_SIZE
ta_kwargs["per_device_eval_batch_size"] = BATCH_SIZE
ta_kwargs["gradient_accumulation_steps"] = GRADIENT_ACCUMULATION_STEPS
ta_kwargs["num_train_epochs"] = NUM_EPOCHS
ta_kwargs["weight_decay"] = 0.01
ta_kwargs.update(pick(("warmup_ratio", 0.1)))
ta_kwargs.update(pick(("fp16", torch.cuda.is_available())))
ta_kwargs.update(pick(("gradient_checkpointing", True)))
ta_kwargs.update(pick(("dataloader_num_workers", 0)))
ta_kwargs.update(pick(("report_to", "none")))
ta_kwargs.update(pick(("run_name", "bert-squad-newsqa-joint")))

args = TrainingArguments(**ta_kwargs)

trainer_params = set(inspect.signature(Trainer.__init__).parameters)
tokenizer_kwarg = "processing_class" if "processing_class" in trainer_params else "tokenizer"

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    **{tokenizer_kwarg: tokenizer},
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [9]:
print("Training on SQuAD + NewsQA combined...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Training on SQuAD + NewsQA combined...


Step,Training Loss
500,2.343492
1000,1.487157
1500,1.354970
2000,1.318020
2500,1.196627
3000,1.201978
3500,1.169280
4000,1.157194
4500,1.098477
5000,1.079941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/CS4248 Group Project/qa_model_joint


## Evaluate on both dev sets

Same inference and decoding logic as `train_qa.ipynb` / `finetune_bert_on_newsqa.ipynb`, run once for SQuAD dev and once for NewsQA dev.

In [10]:
def run_inference(model, dataset, batch_size=BATCH_SIZE):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    eval_set_for_model = dataset.remove_columns(["example_id", "offset_mapping"])
    eval_set_for_model.set_format("torch")

    all_start_logits, all_end_logits = [], []
    with torch.no_grad():
        for i in range(0, len(eval_set_for_model), batch_size):
            batch = eval_set_for_model[i : i + batch_size]
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_start_logits.append(outputs.start_logits.cpu().numpy())
            all_end_logits.append(outputs.end_logits.cpu().numpy())

    return np.concatenate(all_start_logits, axis=0), np.concatenate(all_end_logits, axis=0)


def postprocess_qa_predictions(examples, features, all_start_logits, all_end_logits):
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feature_id]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        context = example["context"]
        feature_indices = features_per_example[example_index]

        valid_answers = []
        for feature_index in feature_indices:
            start_logit = all_start_logits[feature_index]
            end_logit = all_end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -N_BEST - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -N_BEST - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offsets)
                        or end_index >= len(offsets)
                        or offsets[start_index] is None
                        or offsets[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > MAX_ANSWER_LENGTH:
                        continue
                    start_char, end_char = offsets[start_index][0], offsets[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logit[start_index] + end_logit[end_index],
                            "text": context[start_char:end_char],
                        }
                    )

        if valid_answers:
            best = max(valid_answers, key=lambda x: x["score"])
            predictions[example["id"]] = best["text"]
        else:
            predictions[example["id"]] = ""
    return predictions


def score(data_file, predictions_file):
    result = subprocess.run(
        ["python", EVAL_SCRIPT, data_file, predictions_file],
        check=True, capture_output=True, text=True,
    )
    print(result.stdout)

In [11]:
print("Running inference on SQuAD dev...")
squad_start_logits, squad_end_logits = run_inference(model, squad_validation_dataset)
squad_predictions = postprocess_qa_predictions(
    squad_dev_raw, squad_validation_dataset, squad_start_logits, squad_end_logits
)
with open(PREDICTIONS_SQUAD_FILE, "w") as f:
    json.dump(squad_predictions, f)
print("Scoring on SQuAD dev...")
score(SQUAD_DEV_FILE, PREDICTIONS_SQUAD_FILE)

Running inference on SQuAD dev...
Scoring on SQuAD dev...
{
  "exact": 81.11636707663197,
  "f1": 88.57593619884318,
  "total": 10570,
  "HasAns_exact": 81.11636707663197,
  "HasAns_f1": 88.57593619884318,
  "HasAns_total": 10570
}



In [12]:
print("Running inference on NewsQA dev...")
newsqa_start_logits, newsqa_end_logits = run_inference(model, newsqa_validation_dataset)
newsqa_predictions = postprocess_qa_predictions(
    newsqa_dev_raw, newsqa_validation_dataset, newsqa_start_logits, newsqa_end_logits
)
with open(PREDICTIONS_NEWSQA_FILE, "w") as f:
    json.dump(newsqa_predictions, f)
print("Scoring on NewsQA dev...")
score(NEWSQA_DEV_FILE, PREDICTIONS_NEWSQA_FILE)

Running inference on NewsQA dev...
Scoring on NewsQA dev...
{
  "exact": 45.96211365902293,
  "f1": 61.80684533907584,
  "total": 1003,
  "HasAns_exact": 45.96211365902293,
  "HasAns_f1": 61.80684533907584,
  "HasAns_total": 1003
}

